# MEVO data quality and baseline

This notebook is the first Sprint 2 foundation for **MEVO Urban Mobility Analytics**. It verifies the Athena catalog, runs small Athena-to-Pandas smoke queries, and builds a compact inventory of the cleaned datasets.

Athena performs the scans and aggregations; Pandas is used only for the returned analysis-sized results. The cells are designed to be re-run as the S3 dataset grows from the current short history to 30, 60, or more days. This notebook implements the Sprint 2 Data Quality checks and deliberately stops before EDA, business visualizations, or feature engineering.

## Imports

In [1]:
import math
import re
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import boto3
import awswrangler as wr
import pandas as pd
from botocore.exceptions import BotoCoreError, ClientError, NoCredentialsError, PartialCredentialsError
from IPython.display import display

## Configuration

The database and table names below come from the repository's documented Glue/Athena contract. The repository has no tracked workgroup or query-results S3 location, so those values remain unset and Athena uses its configured/default workgroup behavior. Set them only if the deployed workgroup requires an explicit value; do not put credentials or access keys here.

In [2]:
AWS_REGION = "eu-north-1"
ATHENA_DATABASE = "mevo_analytics"
ATHENA_WORKGROUP = None
ATHENA_QUERY_RESULTS = None

# Data-quality assumptions are centralized here so Run All remains reproducible as history grows.
EXPECTED_CADENCE_MINUTES = 10
CADENCE_JITTER_TOLERANCE_MINUTES = 3
SIGNIFICANT_GAP_MULTIPLIER = 1.5
LOCAL_TIMEZONE = "Europe/Warsaw"
# CLEANED year/month/day partitions are Warsaw local calendar dates; RAW partitions are UTC dates.
CLEANED_PARTITION_TIMEZONE = LOCAL_TIMEZONE
CLEANED_REFRESH_HOUR = 3
CLEANED_REFRESH_MINUTE = 0
LAST_REPORTED_FUTURE_TOLERANCE_MINUTES = 15
COVERAGE_PASS_THRESHOLD_PCT = 99.0
COVERAGE_WARN_THRESHOLD_PCT = 95.0

CRITICAL_COLUMNS = {
    "fact_station_status": {
        "snapshot_ts", "station_id", "last_reported",
        "bikes_available", "classic_bikes_available",
        "ebikes_available", "docks_available",
    },
    "dim_station": {
        "snapshot_ts", "station_id", "station_name",
        "latitude", "longitude", "capacity",
    },
}
OPTIONAL_COLUMNS = {"address", "cross_street"}

CLEANED_TABLES = {
    "fact_station_status": {
        "timestamp_column": "snapshot_ts",
        "station_column": "station_id",
        "bike_id_candidates": ("bike_id", "vehicle_id"),
    },
    "dim_station": {
        "timestamp_column": "snapshot_ts",
        "station_column": "station_id",
        "bike_id_candidates": ("bike_id", "vehicle_id"),
    },
}

session = boto3.Session(region_name=AWS_REGION)
sts = session.client("sts")
glue = session.client("glue")


## AWS connection smoke test

This is a read-only identity check. If the local `aws login` session has expired or credentials are unavailable, the cell stops with an actionable error. No credentials, tokens, or secret values are printed.

In [3]:
try:
    identity = sts.get_caller_identity()
except (NoCredentialsError, PartialCredentialsError) as exc:
    raise RuntimeError("AWS credentials are unavailable. Run `aws login` and restart the kernel.") from exc
except (ClientError, BotoCoreError) as exc:
    raise RuntimeError("AWS identity check failed. The local AWS session may have expired; run `aws login` and retry.") from exc

display(pd.DataFrame([{
    "connected": True,
    "region": AWS_REGION,
    "caller_identity_checked": bool(identity.get("Account")),
}]))

,connected,region,caller_identity_checked
0,True,eu-north-1,True


## Athena catalog discovery and verification

In [4]:
try:
    database_metadata = glue.get_database(Name=ATHENA_DATABASE)["Database"]
except (ClientError, BotoCoreError) as exc:
    raise RuntimeError(f"Could not read Glue database {ATHENA_DATABASE!r}.") from exc

catalog_tables = {}
missing_tables = []
for table_name in CLEANED_TABLES:
    try:
        catalog_tables[table_name] = glue.get_table(
            DatabaseName=ATHENA_DATABASE, Name=table_name
        )["Table"]
    except ClientError as exc:
        if exc.response.get("Error", {}).get("Code") == "EntityNotFoundException":
            missing_tables.append(table_name)
        else:
            raise RuntimeError(f"Could not read Glue table {table_name!r}.") from exc
    except BotoCoreError as exc:
        raise RuntimeError(f"Could not read Glue table {table_name!r}.") from exc

table_metadata_rows = []
catalog_columns = {}
catalog_partition_keys = {}
for table_name, table in catalog_tables.items():
    storage = table.get("StorageDescriptor", {})
    parameters = table.get("Parameters", {})
    catalog_columns[table_name] = storage.get("Columns", [])
    catalog_partition_keys[table_name] = table.get("PartitionKeys", [])
    table_metadata_rows.append({
        "database": ATHENA_DATABASE,
        "table": table_name,
        "table_type": table.get("TableType"),
        "location": storage.get("Location"),
        "partition_keys": ", ".join(
            f"{item['Name']}:{item['Type']}" for item in table.get("PartitionKeys", [])
        ),
        "partition_projection": parameters.get("projection.enabled", "false"),
    })

display(pd.DataFrame([
    {"database": database_metadata.get("Name"), "description": database_metadata.get("Description")}
]))
display(pd.DataFrame(table_metadata_rows))

if missing_tables:
    raise RuntimeError(f"Expected CLEANED table(s) not found in {ATHENA_DATABASE!r}: {missing_tables}")

,database,description
0,mevo_analytics,None


,database,table,table_type,location,partition_keys,partition_projection
0,mevo_analytics,fact_station_status,EXTERNAL_TABLE,s3://mevo-analytics-raw-370734990388-eu-north-...,"year:int, month:int, day:int",true
1,mevo_analytics,dim_station,EXTERNAL_TABLE,s3://mevo-analytics-raw-370734990388-eu-north-...,"year:int, month:int, day:int",true


In [5]:
schema_rows = []
for table_name, columns in catalog_columns.items():
    for position, column in enumerate(columns, start=1):
        schema_rows.append({
            "table": table_name,
            "ordinal": position,
            "column": column.get("Name"),
            "athena_type": column.get("Type"),
            "comment": column.get("Comment"),
        })
display(pd.DataFrame(schema_rows))

,table,ordinal,column,athena_type,comment
0,fact_station_status,1,snapshot_ts,timestamp,None
1,fact_station_status,2,feed_last_updated,timestamp,None
2,fact_station_status,3,station_id,string,None
3,fact_station_status,4,last_reported,timestamp,None
4,fact_station_status,5,is_installed,boolean,None
5,fact_station_status,6,is_renting,boolean,None
6,fact_station_status,7,is_returning,boolean,None
7,fact_station_status,8,bikes_available,bigint,None
8,fact_station_status,9,classic_bikes_available,bigint,None
9,fact_station_status,10,ebikes_available,bigint,None


## First Athena → Pandas smoke queries

Each table uses one five-row sample query and one aggregate query. The aggregate is executed in Athena and returns only `COUNT`, timestamp bounds, and the available station/bike distinct counts; no complete table is downloaded.

In [6]:
_IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")

def quote_identifier(identifier):
    if not _IDENTIFIER.fullmatch(identifier):
        raise ValueError(f"Unsafe Athena identifier: {identifier!r}")
    return f'"{identifier}"'

def qualified_table(table_name):
    return f"{quote_identifier(ATHENA_DATABASE)}.{quote_identifier(table_name)}"

def run_athena_query(sql):
    query_kwargs = {
        "database": ATHENA_DATABASE,
        "boto3_session": session,
        "ctas_approach": False,
    }
    if ATHENA_WORKGROUP is not None:
        query_kwargs["workgroup"] = ATHENA_WORKGROUP
    if ATHENA_QUERY_RESULTS is not None:
        query_kwargs["s3_output"] = ATHENA_QUERY_RESULTS
    try:
        return wr.athena.read_sql_query(sql, **query_kwargs)
    except (ClientError, BotoCoreError) as exc:
        raise RuntimeError("Athena query failed; check the AWS session, Glue permissions, and workgroup output configuration.") from exc

samples = {}
inventory_rows = []
for table_name, settings in CLEANED_TABLES.items():
    catalog_column_names = [column["Name"] for column in catalog_columns[table_name]]
    required_columns = {settings["timestamp_column"], settings["station_column"]}
    missing_columns = sorted(required_columns - set(catalog_column_names))
    if missing_columns:
        raise RuntimeError(f"{table_name} is missing expected columns: {missing_columns}")

    select_list = ", ".join(quote_identifier(name) for name in catalog_column_names)
    samples[table_name] = run_athena_query(
        f"SELECT {select_list} FROM {qualified_table(table_name)} LIMIT 5"
    )

    timestamp_column = quote_identifier(settings["timestamp_column"])
    station_column = quote_identifier(settings["station_column"])
    bike_column = next(
        (candidate for candidate in settings["bike_id_candidates"] if candidate in catalog_column_names),
        None,
    )
    distinct_bike_sql = (
        f", COUNT(DISTINCT {quote_identifier(bike_column)}) AS distinct_bikes"
        if bike_column
        else ""
    )
    aggregate_sql = f"""
        SELECT
            COUNT(*) AS row_count,
            MIN({timestamp_column}) AS min_timestamp,
            MAX({timestamp_column}) AS max_timestamp,
            COUNT(DISTINCT {timestamp_column}) AS distinct_snapshots,
            COUNT(DISTINCT {station_column}) AS distinct_stations
            {distinct_bike_sql}
        FROM {qualified_table(table_name)}
    """
    aggregate = run_athena_query(aggregate_sql)
    row = aggregate.iloc[0].to_dict()
    row["table"] = table_name
    row["bike_identifier_column"] = bike_column
    inventory_rows.append(row)

inventory = pd.DataFrame(inventory_rows).set_index("table").reset_index()
inventory["min_timestamp"] = pd.to_datetime(inventory["min_timestamp"], utc=True)
inventory["max_timestamp"] = pd.to_datetime(inventory["max_timestamp"], utc=True)
inventory["covered_days"] = inventory.apply(
    lambda row: (
        pd.NA
        if pd.isna(row["min_timestamp"]) or pd.isna(row["max_timestamp"])
        else max(1, math.ceil((row["max_timestamp"] - row["min_timestamp"]).total_seconds() / 86400))
    ),
    axis=1,
)

display(pd.DataFrame({"table": list(samples), "sample_rows_returned": [len(frame) for frame in samples.values()]}))

,table,sample_rows_returned
0,fact_station_status,5
1,dim_station,5


In [7]:
for table_name, sample in samples.items():
    print(f"{table_name} sample")
    display(sample)

display(inventory[[
    "table",
    "row_count",
    "min_timestamp",
    "max_timestamp",
    "covered_days",
    "distinct_stations",
    "bike_identifier_column",
]])

fact_station_status sample


,snapshot_ts,feed_last_updated,station_id,last_reported,is_installed,is_renting,is_returning,bikes_available,classic_bikes_available,ebikes_available,docks_available
0,2026-08-12 00:52:12.353,2026-08-12 00:51:45,3829,2026-08-12 00:51:57,True,True,True,5,1,4,5
1,2026-08-15 22:08:08.403,2026-08-15 22:07:45,3829,2026-08-15 22:07:58,True,True,True,0,0,0,9
2,2026-08-15 22:08:08.403,2026-08-15 22:07:45,3830,2026-08-15 22:07:58,True,True,True,2,2,0,5
3,2026-08-15 22:08:08.403,2026-08-15 22:07:45,3831,2026-08-15 22:07:58,True,True,True,2,2,0,0
4,2026-08-12 00:52:12.353,2026-08-12 00:51:45,3830,2026-08-12 00:51:57,True,True,True,1,0,1,9


dim_station sample


,snapshot_ts,feed_last_updated,station_id,station_name,address,cross_street,latitude,longitude,capacity,is_virtual_station
0,2026-08-18 01:00:11.994,2026-08-18 01:00:00,3829,GDA001,"plac Dwóch Miast 1, 80-334 Gdańsk","plac Dwóch Miast 1, 80-334 Gdańsk",54.425848,18.577902,10,True
1,2026-08-17 01:00:12.136,2026-08-17 00:59:45,3829,GDA001,"plac Dwóch Miast 1, 80-334 Gdańsk","plac Dwóch Miast 1, 80-334 Gdańsk",54.425848,18.577902,10,True
2,2026-08-14 01:00:12.011,2026-08-14 01:00:00,3829,GDA001,"plac Dwóch Miast 1, 80-334 Gdańsk","plac Dwóch Miast 1, 80-334 Gdańsk",54.425848,18.577902,10,True
3,2026-08-17 01:00:12.136,2026-08-17 00:59:45,3830,GDA002,"Jelitkowska 57-51, 80-342 Gdańsk","Jelitkowska 57-51, 80-342 Gdańsk",54.427905,18.590446,10,True
4,2026-08-17 01:00:12.136,2026-08-17 00:59:45,3831,GDA003,"Pomorska 57a, 80-343 Gdańsk","Pomorska 57a, 80-343 Gdańsk",54.424783,18.595451,10,True


,table,row_count,min_timestamp,max_timestamp,covered_days,distinct_stations,bike_identifier_column
0,fact_station_status,1074568,2026-08-12 00:52:12.353000+00:00,2026-08-20 21:58:08.344000+00:00,9,847,None
1,dim_station,7569,2026-08-12 12:11:16.259000+00:00,2026-08-20 01:00:12.142000+00:00,8,845,None


## Data Quality: temporal coverage and cadence

The expected cadence is approximately 10 minutes, with a 3-minute jitter band. An interval above 15 minutes is treated as a significant gap; missing snapshots are estimated as `ceil(interval / cadence) - 1`. Athena computes timestamp-level aggregates, while Pandas receives only one row per dataset and the completed-period summary.

Coverage ends at the newest Warsaw local day that should already have been published by the daily CLEANED job. The current, not-yet-processed local day is therefore excluded. The Glue partition contract is explicit: CLEANED `year/month/day` uses the Warsaw local calendar date, while RAW partitions use UTC. The pruning dates are derived from the UTC coverage bounds and converted to the local partition timezone, including the local date containing the last instant in the half-open range.

In [8]:
def _as_utc_timestamp(value):
    if value is None or pd.isna(value):
        return pd.NaT
    timestamp = pd.Timestamp(value)
    return timestamp.tz_localize("UTC") if timestamp.tzinfo is None else timestamp.tz_convert("UTC")


def _athena_timestamp_literal(value):
    timestamp = _as_utc_timestamp(value)
    return f"TIMESTAMP '{timestamp.strftime('%Y-%m-%d %H:%M:%S.%f')}'"


def _sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


partition_key_types = {
    table_name: {item["Name"]: item.get("Type", "string") for item in keys}
    for table_name, keys in catalog_partition_keys.items()
}
table_columns = {}
for table_name in CLEANED_TABLES:
    storage_columns = [column["Name"] for column in catalog_columns[table_name]]
    partition_columns = [item["Name"] for item in catalog_partition_keys.get(table_name, [])]
    table_columns[table_name] = list(dict.fromkeys(storage_columns + partition_columns))


def _partition_literal(table_name, column_name, value):
    type_name = partition_key_types.get(table_name, {}).get(column_name, "string").lower()
    if type_name in {"tinyint", "smallint", "int", "integer", "bigint"}:
        return str(value)
    text = f"{value:04d}" if column_name == "year" else f"{value:02d}"
    return _sql_string(text)


def _partition_date_comparison(table_name, boundary, operator):
    required = {"year", "month", "day"}
    if not required.issubset(partition_key_types.get(table_name, {})):
        return "TRUE"
    year = quote_identifier("year")
    month = quote_identifier("month")
    day = quote_identifier("day")
    y_value = _partition_literal(table_name, "year", boundary.year)
    m_value = _partition_literal(table_name, "month", boundary.month)
    d_value = _partition_literal(table_name, "day", boundary.day)
    if operator == ">=":
        return f"(({year} > {y_value}) OR ({year} = {y_value} AND {month} > {m_value}) OR ({year} = {y_value} AND {month} = {m_value} AND {day} >= {d_value}))"
    if operator == "<=":
        return f"(({year} < {y_value}) OR ({year} = {y_value} AND {month} < {m_value}) OR ({year} = {y_value} AND {month} = {m_value} AND {day} <= {d_value}))"
    raise ValueError(f"Unsupported partition comparison: {operator}")


def _partition_date_predicate(table_name, start_date, end_date):
    if start_date > end_date:
        return "FALSE"
    return (
        f"{_partition_date_comparison(table_name, start_date, '>=')} AND "
        f"{_partition_date_comparison(table_name, end_date, '<=')}"
    )


def _partition_dates_for_utc_bounds(start_utc, end_utc):
    """Return inclusive CLEANED partition dates covering a UTC half-open range."""
    start_utc = _as_utc_timestamp(start_utc)
    end_utc = _as_utc_timestamp(end_utc)
    if pd.isna(start_utc) or pd.isna(end_utc) or end_utc <= start_utc:
        raise ValueError("UTC coverage bounds must be non-empty and ordered")
    partition_timezone = ZoneInfo(CLEANED_PARTITION_TIMEZONE)
    last_included_utc = end_utc - pd.Timedelta(microseconds=1)
    return (
        start_utc.tz_convert(partition_timezone).date(),
        last_included_utc.tz_convert(partition_timezone).date(),
    )


local_timezone = ZoneInfo(LOCAL_TIMEZONE)
now_local = datetime.now(local_timezone)
refresh_today = now_local.replace(
    hour=CLEANED_REFRESH_HOUR, minute=CLEANED_REFRESH_MINUTE, second=0, microsecond=0
)
latest_expected_local_date = (
    now_local.date() - timedelta(days=1 if now_local >= refresh_today else 2)
)
cadence_lower_bound = max(0, EXPECTED_CADENCE_MINUTES - CADENCE_JITTER_TOLERANCE_MINUTES)
cadence_upper_bound = EXPECTED_CADENCE_MINUTES + CADENCE_JITTER_TOLERANCE_MINUTES
significant_gap_threshold = max(
    cadence_upper_bound, EXPECTED_CADENCE_MINUTES * SIGNIFICANT_GAP_MULTIPLIER
)


def _cadence_query(table_name):
    timestamp_column = quote_identifier(CLEANED_TABLES[table_name]["timestamp_column"])
    table = qualified_table(table_name)
    return f"""
        WITH snapshots AS (
            SELECT DISTINCT CAST({timestamp_column} AS timestamp) AS snapshot_ts
            FROM {table}
            WHERE {timestamp_column} IS NOT NULL
        ),
        ordered AS (
            SELECT snapshot_ts, LAG(snapshot_ts) OVER (ORDER BY snapshot_ts) AS previous_snapshot_ts
            FROM snapshots
        ),
        gaps AS (
            SELECT date_diff('second', previous_snapshot_ts, snapshot_ts) / 60.0 AS interval_minutes
            FROM ordered
            WHERE previous_snapshot_ts IS NOT NULL
        )
        SELECT
            (SELECT COUNT(*) FROM snapshots) AS distinct_snapshot_count,
            MIN(interval_minutes) AS min_interval_minutes,
            AVG(interval_minutes) AS mean_interval_minutes,
            approx_percentile(interval_minutes, 0.50) AS median_interval_minutes,
            MAX(interval_minutes) AS max_interval_minutes,
            approx_percentile(interval_minutes, 0.95) AS p95_interval_minutes,
            COALESCE(SUM(CASE WHEN interval_minutes < {cadence_lower_bound} OR interval_minutes > {cadence_upper_bound} THEN 1 ELSE 0 END), 0) AS intervals_outside_tolerance,
            COALESCE(SUM(CASE WHEN interval_minutes > {significant_gap_threshold} THEN 1 ELSE 0 END), 0) AS significant_gap_count,
            COALESCE(SUM(CASE WHEN interval_minutes > {significant_gap_threshold} THEN CAST(CEIL(interval_minutes / {EXPECTED_CADENCE_MINUTES}) AS BIGINT) - 1 ELSE 0 END), 0) AS estimated_missing_snapshots_in_large_gaps
        FROM gaps
    """


cadence_rows = []
for table_name in ("fact_station_status",):
    row = run_athena_query(_cadence_query(table_name)).iloc[0].to_dict()
    row["table"] = table_name
    cadence_rows.append(row)
cadence_summary = pd.DataFrame(cadence_rows)[[
    "table", "distinct_snapshot_count", "median_interval_minutes",
    "mean_interval_minutes", "min_interval_minutes", "max_interval_minutes",
    "p95_interval_minutes", "intervals_outside_tolerance",
    "significant_gap_count", "estimated_missing_snapshots_in_large_gaps",
]].sort_values("table")
display(cadence_summary)
print(
    f"Cadence tolerance: {cadence_lower_bound:.0f}-{cadence_upper_bound:.0f} minutes; "
    f"significant gap > {significant_gap_threshold:.1f} minutes."
)


fact_inventory = inventory.set_index("table").loc["fact_station_status"]
first_fact_snapshot_utc = _as_utc_timestamp(fact_inventory["min_timestamp"])
coverage_start_local_date = first_fact_snapshot_utc.tz_convert(LOCAL_TIMEZONE).date() if not pd.isna(first_fact_snapshot_utc) else None
coverage_end_local_date = latest_expected_local_date
coverage_rows = []
if coverage_start_local_date is None or coverage_start_local_date > coverage_end_local_date:
    coverage_rows.append({
        "actual_snapshots": 0, "expected_snapshots": 0,
        "estimated_missing_snapshots": 0, "coverage_pct": pd.NA,
        "period_start_local": coverage_start_local_date,
        "period_end_local": coverage_end_local_date,
        "status": "WARN",
        "notes": "No completed CLEANED local day is available for the coverage calculation.",
    })
else:
    coverage_end_local_midnight = datetime.combine(
        coverage_end_local_date + timedelta(days=1), datetime.min.time(), tzinfo=local_timezone
    )
    coverage_start_utc = first_fact_snapshot_utc
    coverage_end_utc = pd.Timestamp(coverage_end_local_midnight).tz_convert("UTC")
    coverage_partition_start_date, coverage_partition_end_date = _partition_dates_for_utc_bounds(
        coverage_start_utc, coverage_end_utc
    )
    coverage_partition_filter = _partition_date_predicate(
        "fact_station_status", coverage_partition_start_date, coverage_partition_end_date
    )
    coverage_query = f"""
        SELECT
            COUNT(DISTINCT {quote_identifier('snapshot_ts')}) AS actual_snapshots,
            MIN({quote_identifier('snapshot_ts')}) AS actual_period_start_utc,
            MAX({quote_identifier('snapshot_ts')}) AS actual_period_end_utc
        FROM {qualified_table('fact_station_status')}
        WHERE {quote_identifier('snapshot_ts')} >= {_athena_timestamp_literal(coverage_start_utc)}
          AND {quote_identifier('snapshot_ts')} < {_athena_timestamp_literal(coverage_end_utc)}
          AND {coverage_partition_filter}
    """
    coverage_actual = run_athena_query(coverage_query).iloc[0]
    duration_minutes = (coverage_end_utc - coverage_start_utc).total_seconds() / 60
    expected_snapshots = max(0, math.ceil(duration_minutes / EXPECTED_CADENCE_MINUTES))
    actual_snapshots = int(coverage_actual["actual_snapshots"] or 0)
    estimated_missing_snapshots = max(expected_snapshots - actual_snapshots, 0)
    coverage_pct = (100 * actual_snapshots / expected_snapshots) if expected_snapshots else pd.NA
    if pd.isna(coverage_pct):
        coverage_status = "WARN"
    elif coverage_pct >= COVERAGE_PASS_THRESHOLD_PCT:
        coverage_status = "PASS"
    elif coverage_pct >= COVERAGE_WARN_THRESHOLD_PCT:
        coverage_status = "WARN"
    else:
        coverage_status = "FAIL"
    coverage_rows.append({
        "actual_snapshots": actual_snapshots,
        "expected_snapshots": expected_snapshots,
        "estimated_missing_snapshots": estimated_missing_snapshots,
        "coverage_pct": coverage_pct,
        "period_start_local": coverage_start_local_date,
        "period_end_local": coverage_end_local_date,
        "status": coverage_status,
        "notes": (
            "Expected slots use the first observed snapshot through the end of the newest completed Warsaw day; "
            "the current unprocessed day is excluded."
        ),
    })
coverage_summary = pd.DataFrame(coverage_rows)
display(coverage_summary)

,table,distinct_snapshot_count,median_interval_minutes,mean_interval_minutes,min_interval_minutes,max_interval_minutes,p95_interval_minutes,intervals_outside_tolerance,significant_gap_count,estimated_missing_snapshots_in_large_gaps
0,fact_station_status,1277.0,9.990269,10.011821,9.933333,25.983333,10.0,2.0,2.0,3.0


Cadence tolerance: 7-13 minutes; significant gap > 15.0 minutes.


,actual_snapshots,expected_snapshots,estimated_missing_snapshots,coverage_pct,period_start_local,period_end_local,status,notes
0,1277,1279,2,99.843628,2026-08-12,2026-08-20,PASS,Expected slots use the first observed snapshot...


## Data Quality: duplicates and NULLs

Duplicate checks use the declared grain `(snapshot_ts, station_id)`. `duplicate_row_count` means excess rows beyond the first row per duplicate key, and `duplicate_pct` uses total table rows as denominator. NULL counts are computed for every storage and partition column; critical fields are separated from optional descriptive fields such as address and cross street.

In [9]:
duplicate_metrics_rows = []
duplicate_samples = {}
for table_name, settings in CLEANED_TABLES.items():
    timestamp_column = quote_identifier(settings["timestamp_column"])
    station_column = quote_identifier(settings["station_column"])
    duplicate_sql = f"""
        WITH key_counts AS (
            SELECT {timestamp_column} AS snapshot_ts, {station_column} AS station_id, COUNT(*) AS rows_per_key
            FROM {qualified_table(table_name)}
            GROUP BY {timestamp_column}, {station_column}
        )
        SELECT
            COALESCE(SUM(rows_per_key - 1), 0) AS duplicate_row_count,
            COALESCE(SUM(CASE WHEN rows_per_key > 1 THEN 1 ELSE 0 END), 0) AS duplicate_key_count,
            COALESCE(SUM(CASE WHEN rows_per_key > 1 THEN rows_per_key ELSE 0 END), 0) AS rows_in_duplicate_keys
        FROM key_counts
    """
    duplicate_row = run_athena_query(duplicate_sql).iloc[0].to_dict()
    duplicate_row["table"] = table_name
    duplicate_row["row_count"] = int(inventory.set_index("table").loc[table_name, "row_count"])
    duplicate_metrics_rows.append(duplicate_row)
    if int(duplicate_row["duplicate_key_count"] or 0) > 0:
        duplicate_samples[table_name] = run_athena_query(f"""
            SELECT {timestamp_column} AS snapshot_ts, {station_column} AS station_id, COUNT(*) AS duplicate_rows
            FROM {qualified_table(table_name)}
            GROUP BY {timestamp_column}, {station_column}
            HAVING COUNT(*) > 1
            ORDER BY duplicate_rows DESC
            LIMIT 20
        """)
    else:
        duplicate_samples[table_name] = pd.DataFrame(columns=["snapshot_ts", "station_id", "duplicate_rows"])

duplicate_metrics = pd.DataFrame(duplicate_metrics_rows)
duplicate_metrics["duplicate_pct"] = duplicate_metrics.apply(
    lambda row: (100 * row["duplicate_row_count"] / row["row_count"]) if row["row_count"] else 0,
    axis=1,
)
display(duplicate_metrics[[
    "table", "row_count", "duplicate_row_count", "duplicate_key_count",
    "rows_in_duplicate_keys", "duplicate_pct",
]])
for table_name, sample in duplicate_samples.items():
    if not sample.empty:
        print(f"{table_name}: sample of duplicate keys")
        display(sample)


null_analysis_rows = []
for table_name in CLEANED_TABLES:
    columns = table_columns[table_name]
    null_expressions = [
        f"SUM(CASE WHEN {quote_identifier(column)} IS NULL THEN 1 ELSE 0 END) AS null_{position}"
        for position, column in enumerate(columns)
    ]
    null_query = f"SELECT COUNT(*) AS row_count, {', '.join(null_expressions)} FROM {qualified_table(table_name)}"
    null_row = run_athena_query(null_query).iloc[0].to_dict()
    row_count = int(null_row["row_count"] or 0)
    for position, column in enumerate(columns):
        null_count = int(null_row.get(f"null_{position}", 0) or 0)
        role = (
            "critical" if column in CRITICAL_COLUMNS.get(table_name, {})
            else "optional" if column in OPTIONAL_COLUMNS
            else "non-critical"
        )
        null_analysis_rows.append({
            "table": table_name,
            "column": column,
            "row_count": row_count,
            "null_count": null_count,
            "null_pct": (100 * null_count / row_count) if row_count else pd.NA,
            "role": role,
            "interpretation": (
                "NULL requires investigation" if role == "critical"
                else "NULL can be valid for this optional/descriptive field"
            ),
        })
null_analysis = pd.DataFrame(null_analysis_rows)
display(null_analysis[[
    "table", "column", "null_count", "null_pct", "role", "interpretation",
]])

,table,row_count,duplicate_row_count,duplicate_key_count,rows_in_duplicate_keys,duplicate_pct
0,fact_station_status,1074568,0,0,0,0.0
1,dim_station,7569,0,0,0,0.0


,table,column,null_count,null_pct,role,interpretation
0,fact_station_status,snapshot_ts,0,0.0,critical,NULL requires investigation
1,fact_station_status,feed_last_updated,0,0.0,non-critical,NULL can be valid for this optional/descriptiv...
2,fact_station_status,station_id,0,0.0,critical,NULL requires investigation
3,fact_station_status,last_reported,0,0.0,critical,NULL requires investigation
4,fact_station_status,is_installed,0,0.0,non-critical,NULL can be valid for this optional/descriptiv...
5,fact_station_status,is_renting,0,0.0,non-critical,NULL can be valid for this optional/descriptiv...
6,fact_station_status,is_returning,0,0.0,non-critical,NULL can be valid for this optional/descriptiv...
7,fact_station_status,bikes_available,0,0.0,critical,NULL requires investigation
8,fact_station_status,classic_bikes_available,0,0.0,critical,NULL requires investigation
9,fact_station_status,ebikes_available,0,0.0,critical,NULL requires investigation


## Data Quality: logical and range validation

Negative counters, impossible coordinates, empty required identifiers, malformed booleans, and materially future `last_reported` values are failures. The vehicle-total composition check is reported as WARN because the relationship may depend on how GBFS exposes vehicle subtypes and is therefore interpretive rather than an unconditional error.

In [10]:
column_types = {
    table_name: {column["Name"]: column.get("Type", "").lower() for column in columns}
    for table_name, columns in catalog_columns.items()
}


def _quality_alias(check_name):
    return "invalid_" + re.sub(r"[^A-Za-z0-9_]", "_", check_name)


quality_specs = {
    "fact_station_status": [
        ("bikes_available_nonnegative", ["bikes_available"], lambda q: f"{q('bikes_available')} < 0", "FAIL", "bikes_available must be non-negative."),
        ("classic_bikes_available_nonnegative", ["classic_bikes_available"], lambda q: f"{q('classic_bikes_available')} < 0", "FAIL", "classic_bikes_available must be non-negative."),
        ("ebikes_available_nonnegative", ["ebikes_available"], lambda q: f"{q('ebikes_available')} < 0", "FAIL", "ebikes_available must be non-negative."),
        ("docks_available_nonnegative", ["docks_available"], lambda q: f"{q('docks_available')} < 0", "FAIL", "docks_available must be non-negative."),
        ("vehicle_total_consistency", ["bikes_available", "classic_bikes_available", "ebikes_available"], lambda q: f"{q('classic_bikes_available')} + {q('ebikes_available')} <> {q('bikes_available')}", "WARN", "Interpretive GBFS subtype-total relationship; review before treating as an error."),
        ("last_reported_not_materially_future", ["snapshot_ts", "last_reported"], lambda q: f"date_diff('second', {q('snapshot_ts')}, {q('last_reported')}) > {LAST_REPORTED_FUTURE_TOLERANCE_MINUTES * 60}", "FAIL", f"last_reported may be at most {LAST_REPORTED_FUTURE_TOLERANCE_MINUTES} minutes after snapshot_ts."),
    ],
    "dim_station": [
        ("capacity_nonnegative", ["capacity"], lambda q: f"{q('capacity')} < 0", "FAIL", "capacity must be non-negative."),
        ("latitude_in_range", ["latitude"], lambda q: f"{q('latitude')} < -90 OR {q('latitude')} > 90", "FAIL", "latitude must be in [-90, 90]."),
        ("longitude_in_range", ["longitude"], lambda q: f"{q('longitude')} < -180 OR {q('longitude')} > 180", "FAIL", "longitude must be in [-180, 180]."),
        ("station_id_nonempty", ["station_id"], lambda q: f"trim(CAST({q('station_id')} AS varchar)) = ''", "FAIL", "station_id must not be empty."),
        ("station_name_nonempty", ["station_name"], lambda q: f"trim(CAST({q('station_name')} AS varchar)) = ''", "FAIL", "station_name must not be empty."),
    ],
}
boolean_columns = {
    "fact_station_status": ["is_installed", "is_renting", "is_returning"],
    "dim_station": ["is_virtual_station"],
}
logical_rows = []
for table_name in CLEANED_TABLES:
    available_columns = {column["Name"] for column in catalog_columns[table_name]}
    available_columns |= {item["Name"] for item in catalog_partition_keys.get(table_name, [])}
    expressions = []
    expression_specs = []
    for check_name, columns, condition_builder, status_if_issue, notes in quality_specs[table_name]:
        if set(columns).issubset(available_columns):
            condition = condition_builder(quote_identifier)
            alias = _quality_alias(check_name)
            expressions.append(f"SUM(CASE WHEN {' AND '.join(f'{quote_identifier(column)} IS NOT NULL' for column in columns)} AND ({condition}) THEN 1 ELSE 0 END) AS {alias}")
            expression_specs.append((check_name, columns, alias, status_if_issue, notes))
        else:
            missing = sorted(set(columns) - available_columns)
            logical_rows.append({
                "table": table_name, "check": check_name, "invalid_count": pd.NA,
                "invalid_pct": pd.NA, "status": "FAIL",
                "notes": f"Required column(s) missing from Glue schema: {missing}.",
            })
    for column in boolean_columns[table_name]:
        if column not in available_columns:
            logical_rows.append({
                "table": table_name, "check": f"{column}_boolean", "invalid_count": pd.NA,
                "invalid_pct": pd.NA, "status": "FAIL",
                "notes": f"Required boolean column missing from Glue schema: {column}.",
            })
            continue
        alias = _quality_alias(f"{column}_boolean")
        type_name = column_types[table_name].get(column, "")
        if type_name == "boolean":
            expressions.append(f"CAST(0 AS BIGINT) AS {alias}")
            notes = "Glue/Athena schema is boolean; non-boolean values cannot be stored in this column."
        elif "char" in type_name or "string" in type_name or "varchar" in type_name:
            expressions.append(
                f"SUM(CASE WHEN {quote_identifier(column)} IS NOT NULL AND lower(trim(CAST({quote_identifier(column)} AS varchar))) NOT IN ('true', 'false') THEN 1 ELSE 0 END) AS {alias}"
            )
            notes = "String-backed boolean values are accepted only when they are true/false."
        else:
            expressions.append(f"SUM(CASE WHEN {quote_identifier(column)} IS NOT NULL THEN 1 ELSE 0 END) AS {alias}")
            notes = f"Unexpected Glue/Athena type {type_name!r} for a boolean column."
        expression_specs.append((f"{column}_boolean", [column], alias, "FAIL", notes))
    validation_query = f"SELECT COUNT(*) AS row_count{', ' + ', '.join(expressions) if expressions else ''} FROM {qualified_table(table_name)}"
    validation_row = run_athena_query(validation_query).iloc[0].to_dict()
    row_count = int(validation_row["row_count"] or 0)
    for check_name, columns, alias, status_if_issue, notes in expression_specs:
        invalid_count = int(validation_row.get(alias, 0) or 0)
        logical_rows.append({
            "table": table_name, "check": check_name,
            "invalid_count": invalid_count,
            "invalid_pct": (100 * invalid_count / row_count) if row_count else pd.NA,
            "status": status_if_issue if invalid_count else "PASS",
            "notes": notes,
        })
logical_validation = pd.DataFrame(logical_rows)
display(logical_validation)

,table,check,invalid_count,invalid_pct,status,notes
0,fact_station_status,bikes_available_nonnegative,0,0.0,PASS,bikes_available must be non-negative.
1,fact_station_status,classic_bikes_available_nonnegative,0,0.0,PASS,classic_bikes_available must be non-negative.
2,fact_station_status,ebikes_available_nonnegative,0,0.0,PASS,ebikes_available must be non-negative.
3,fact_station_status,docks_available_nonnegative,0,0.0,PASS,docks_available must be non-negative.
4,fact_station_status,vehicle_total_consistency,0,0.0,PASS,Interpretive GBFS subtype-total relationship; ...
5,fact_station_status,last_reported_not_materially_future,0,0.0,PASS,last_reported may be at most 15 minutes after ...
6,fact_station_status,is_installed_boolean,0,0.0,PASS,Glue/Athena schema is boolean; non-boolean val...
7,fact_station_status,is_renting_boolean,0,0.0,PASS,Glue/Athena schema is boolean; non-boolean val...
8,fact_station_status,is_returning_boolean,0,0.0,PASS,Glue/Athena schema is boolean; non-boolean val...
9,dim_station,capacity_nonnegative,0,0.0,PASS,capacity must be non-negative.


## Data Quality: records per fact snapshot

Athena first reduces the fact table to one station count per snapshot. The resulting statistics are diagnostic only: station availability can naturally vary when stations are offline, serviced, or temporarily stop reporting. No records-per-snapshot threshold contributes to the Dataset Health status.

In [11]:
fact_snapshot_counts = run_athena_query(f"""
    SELECT {quote_identifier('snapshot_ts')} AS snapshot_ts, COUNT(*) AS station_count
    FROM {qualified_table('fact_station_status')}
    GROUP BY {quote_identifier('snapshot_ts')}
    ORDER BY snapshot_ts
""")
fact_snapshot_counts["station_count"] = pd.to_numeric(fact_snapshot_counts["station_count"])
if fact_snapshot_counts.empty:
    records_per_snapshot_stats = pd.DataFrame(columns=["metric", "value"])
else:
    counts = fact_snapshot_counts["station_count"]
    median_count = counts.median()
    mad = (counts - median_count).abs().median()
    records_per_snapshot_stats = pd.DataFrame([
        {"metric": "min", "value": counts.min()},
        {"metric": "max", "value": counts.max()},
        {"metric": "mean", "value": counts.mean()},
        {"metric": "median", "value": median_count},
        {"metric": "std", "value": counts.std(ddof=0)},
        {"metric": "mad", "value": mad},
        {"metric": "p01", "value": counts.quantile(0.01)},
        {"metric": "p05", "value": counts.quantile(0.05)},
        {"metric": "p95", "value": counts.quantile(0.95)},
        {"metric": "p99", "value": counts.quantile(0.99)},
    ])
display(records_per_snapshot_stats)
print("Informational diagnostic only: station-count variation per snapshot is expected in this domain and does not affect Dataset Health.")

,metric,value
0,min,837.000000
1,max,845.000000
2,mean,841.478465
3,median,842.000000
4,std,2.136797
5,mad,1.000000
6,p01,837.000000
7,p05,837.000000
8,p95,843.000000
9,p99,845.000000


Informational diagnostic only: station-count variation per snapshot is expected in this domain and does not affect Dataset Health.


## Data Quality: dim_station cadence and grain

The dimension is not assumed to be 10-minute data. Athena returns one count per reference snapshot; Pandas then summarizes the observed intervals and station counts. This makes the current row volume interpretable as approximately `reference snapshots × stations per reference snapshot`.

In [12]:
dim_snapshot_counts = run_athena_query(f"""
    SELECT {quote_identifier('snapshot_ts')} AS snapshot_ts, COUNT(*) AS station_count
    FROM {qualified_table('dim_station')}
    GROUP BY {quote_identifier('snapshot_ts')}
    ORDER BY snapshot_ts
""")
dim_snapshot_counts["station_count"] = pd.to_numeric(dim_snapshot_counts["station_count"])
dim_snapshot_counts["snapshot_ts"] = pd.to_datetime(dim_snapshot_counts["snapshot_ts"], utc=True)
dim_intervals = dim_snapshot_counts["snapshot_ts"].sort_values().diff().dt.total_seconds().div(60).dropna()
if dim_snapshot_counts.empty:
    dim_cadence_summary = pd.DataFrame()
    dim_grain_summary = pd.DataFrame()
else:
    dim_cadence_summary = pd.DataFrame([{
        "dim_snapshot_count": len(dim_snapshot_counts),
        "min_interval_minutes": dim_intervals.min() if not dim_intervals.empty else pd.NA,
        "mean_interval_minutes": dim_intervals.mean() if not dim_intervals.empty else pd.NA,
        "median_interval_minutes": dim_intervals.median() if not dim_intervals.empty else pd.NA,
        "max_interval_minutes": dim_intervals.max() if not dim_intervals.empty else pd.NA,
        "p95_interval_minutes": dim_intervals.quantile(0.95) if not dim_intervals.empty else pd.NA,
    }])
    dim_counts = dim_snapshot_counts["station_count"]
    dim_row_count = int(inventory.set_index("table").loc["dim_station", "row_count"])
    typical_station_count = dim_counts.median()
    dim_grain_summary = pd.DataFrame([{
        "dim_row_count": dim_row_count,
        "dim_snapshot_count": len(dim_snapshot_counts),
        "min_stations_per_snapshot": dim_counts.min(),
        "max_stations_per_snapshot": dim_counts.max(),
        "mean_stations_per_snapshot": dim_counts.mean(),
        "median_stations_per_snapshot": typical_station_count,
        "estimated_rows_from_median": len(dim_snapshot_counts) * typical_station_count,
        "actual_rows_per_snapshot": dim_row_count / len(dim_snapshot_counts),
    }])
display(dim_snapshot_counts)
display(dim_cadence_summary)
display(dim_grain_summary)

,snapshot_ts,station_count
0,2026-08-12 12:11:16.259000+00:00,837
1,2026-08-13 01:00:12.126000+00:00,837
2,2026-08-14 01:00:12.011000+00:00,843
3,2026-08-15 01:00:12.082000+00:00,841
4,2026-08-16 01:00:12.259000+00:00,842
5,2026-08-17 01:00:12.136000+00:00,842
6,2026-08-18 01:00:11.994000+00:00,841
7,2026-08-19 01:00:12.146000+00:00,843
8,2026-08-20 01:00:12.142000+00:00,843


,dim_snapshot_count,min_interval_minutes,mean_interval_minutes,median_interval_minutes,max_interval_minutes,p95_interval_minutes
0,9,768.931117,1356.116423,1439.999008,1440.00295,1440.002804


,dim_row_count,dim_snapshot_count,min_stations_per_snapshot,max_stations_per_snapshot,mean_stations_per_snapshot,median_stations_per_snapshot,estimated_rows_from_median,actual_rows_per_snapshot
0,7569,9,837,843,841.0,842.0,7578.0,841.0


## Data Quality: freshness and dataset health summary

Freshness is evaluated against the scheduler contract, not against the wall-clock timestamp of the newest row. Before the configured refresh time, the newest expected local day is two days behind today; after refresh, it is yesterday. A one-day lag is WARN and a lag of two or more days is FAIL. The final table uses transparent PASS/WARN/FAIL rules and does not compress them into a composite score.

In [13]:
freshness_rows = []
freshness_rank = {"PASS": 0, "WARN": 1, "FAIL": 2}
for table_name in CLEANED_TABLES:
    latest_timestamp = _as_utc_timestamp(
        inventory.set_index("table").loc[table_name, "max_timestamp"]
    )
    if pd.isna(latest_timestamp):
        actual_latest_local_date = None
        lag_days = pd.NA
        status = "FAIL"
        notes = "No observed snapshot timestamp."
    else:
        actual_latest_local_date = latest_timestamp.tz_convert(LOCAL_TIMEZONE).date()
        lag_days = (latest_expected_local_date - actual_latest_local_date).days
        if lag_days <= 0:
            status = "PASS"
            notes = "Latest observed local day is at or ahead of the newest day expected from the scheduler."
        elif lag_days == 1:
            status = "WARN"
            notes = "One completed local day is behind the scheduler expectation."
        else:
            status = "FAIL"
            notes = "At least two completed local days are behind the scheduler expectation."
    freshness_rows.append({
        "table": table_name,
        "observed_at_local": now_local.isoformat(timespec="minutes"),
        "expected_latest_local_date": latest_expected_local_date,
        "actual_latest_snapshot_local_date": actual_latest_local_date,
        "lag_days": lag_days,
        "status": status,
        "notes": notes,
    })
freshness = pd.DataFrame(freshness_rows)
display(freshness)
freshness_status = max(freshness["status"], key=lambda status: freshness_rank[status]) if not freshness.empty else "FAIL"


def _count_or_zero(frame, column):
    if frame.empty or column not in frame:
        return 0
    return int(pd.to_numeric(frame[column], errors="coerce").fillna(0).sum())


coverage_row = coverage_summary.iloc[0]
coverage_status = coverage_row["status"]
duplicate_key_count = _count_or_zero(duplicate_metrics, "duplicate_key_count")
critical_nulls = null_analysis[(null_analysis["role"] == "critical") & (null_analysis["null_count"] > 0)]
critical_null_count = int(critical_nulls["null_count"].sum()) if not critical_nulls.empty else 0
range_failures = logical_validation[logical_validation["status"] == "FAIL"]
range_warnings = logical_validation[logical_validation["status"] == "WARN"]
if not range_failures.empty:
    range_status = "FAIL"
elif not range_warnings.empty:
    range_status = "WARN"
else:
    range_status = "PASS"
estimated_missing = coverage_row["estimated_missing_snapshots"]
estimated_missing_status = "PASS" if int(estimated_missing or 0) == 0 else coverage_status if coverage_status == "FAIL" else "WARN"
duplicate_status = "FAIL" if duplicate_key_count else "PASS"
critical_null_status = "FAIL" if critical_null_count else "PASS"
dim_cadence_status = "PASS" if len(dim_snapshot_counts) >= 1 else "FAIL"

health_summary = pd.DataFrame([
    {"metric": "temporal coverage", "value": coverage_row["coverage_pct"], "status": coverage_status, "notes": coverage_row["notes"]},
    {"metric": "estimated missing snapshots", "value": estimated_missing, "status": estimated_missing_status, "notes": "Expected minus actual DISTINCT snapshots in the completed period."},
    {"metric": "duplicate keys", "value": duplicate_key_count, "status": duplicate_status, "notes": "Duplicate grain: (snapshot_ts, station_id); samples are shown above when present."},
    {"metric": "critical NULLs", "value": critical_null_count, "status": critical_null_status, "notes": "Optional address/cross_street NULLs are not included in this failure count."},
    {"metric": "invalid ranges and logic", "value": int(len(range_failures)), "status": range_status, "notes": "WARN may represent the interpretive vehicle-total consistency check."},
    {"metric": "dim_station cadence", "value": int(len(dim_snapshot_counts)), "status": dim_cadence_status, "notes": "Observed reference snapshots and their intervals are shown above; no cadence was assumed."},
    {"metric": "freshness", "value": freshness_status, "status": freshness_status, "notes": "Compared with the newest local day expected after the D+1 refresh."},
])
display(health_summary)
anomalies = health_summary[health_summary["status"] != "PASS"]
if anomalies.empty:
    print("Anomalies found: none under the explicit rules above.")
else:
    print("Anomalies found (including WARN conditions):")
    display(anomalies)

,table,observed_at_local,expected_latest_local_date,actual_latest_snapshot_local_date,lag_days,status,notes
0,fact_station_status,2026-08-21T17:23+02:00,2026-08-20,2026-08-20,0,PASS,Latest observed local day is at or ahead of th...
1,dim_station,2026-08-21T17:23+02:00,2026-08-20,2026-08-20,0,PASS,Latest observed local day is at or ahead of th...


,metric,value,status,notes
0,temporal coverage,99.843628,PASS,Expected slots use the first observed snapshot...
1,estimated missing snapshots,2,WARN,Expected minus actual DISTINCT snapshots in th...
2,duplicate keys,0,PASS,"Duplicate grain: (snapshot_ts, station_id); sa..."
3,critical NULLs,0,PASS,Optional address/cross_street NULLs are not in...
4,invalid ranges and logic,0,PASS,WARN may represent the interpretive vehicle-to...
5,dim_station cadence,9,PASS,Observed reference snapshots and their interva...
6,freshness,PASS,PASS,Compared with the newest local day expected af...


Anomalies found (including WARN conditions):


,metric,value,status,notes
1,estimated missing snapshots,2,WARN,Expected minus actual DISTINCT snapshots in th...
